In [2]:
from pathlib import Path

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

True

In [4]:
PDF_PATH = Path("../documents/hr_policy.pdf")

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

embeddings = MistralAIEmbeddings(
    model="mistral-embed"
)

In [10]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(type(vector_store))
print(vector_store.index)
print(f"Number of vectors: {vector_store.index.ntotal}")

<class 'langchain_community.vectorstores.faiss.FAISS'>
<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x10a0ddaf0> >
Number of vectors: 70


In [16]:
query = "How many paid leaves can an employee take?"

results = vector_store.similarity_search(
    query,
    k=1,
)

for i, doc in enumerate(results):
    print(f"\n{'=' * 60}")
    print(f"RESULT {i + 1}")
    print(f"{'=' * 60}")
    print(doc.page_content)
    print(f"\nMetadata: {doc.metadata}")


RESULT 1
Index of Contents 
1. Document Revision History ..................................................................................................................... 3 
General Statement .......................................................................................................................................... 3 
2. Attendance Policy ................................................................................................................................... 3

Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-13T09:38:48+00:00', 'author': 'Invx28', 'moddate': '2023-04-13T15:09:50+05:30', 'source': '../documents/hr_policy.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2'}


In [12]:
results_with_scores = vector_store.similarity_search_with_score(
    query,
    k=3,
)

for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n{'=' * 60}")
    print(f"RESULT {i + 1}")
    print(f"{'=' * 60}")
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")


RESULT 1
Score: 0.4167328476905823
Content: ii. Total 12 days of Earned leaves will be granted per year.(Year considered as financial year from April- 
March) 
iii. Employee will get Earned leaves once they get complete probation period, it will be based on 
financial year. 
iv. The employer has rights to terminate Employee(s) without notice and without any payment in lieu  
of notice if Employee is taking more than 13 unplanned leaves (including weekends and official offs) 
without any proper updates (and proofs).
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-13T09:38:48+00:00', 'author': 'Invx28', 'moddate': '2023-04-13T15:09:50+05:30', 'source': '../documents/hr_policy.pdf', 'total_pages': 16, 'page': 5, 'page_label': '6'}

RESULT 2
Score: 0.42156946659088135
Content: given time should not be less than 2:3. 
14. Employees going on leave have to explain/delegate their responsibility to another team member. 
15. In case an empl

## Observations

- A vector store is used to store and search embedding vectors.
- FAISS is a similarity-search library that can efficiently search
  vectors.
- LangChain provides a FAISS vector store integration that connects
  FAISS with LangChain Documents and embeddings.
- Each chunk is converted into an embedding vector.
- The vector store maintains the relationship between vectors and
  their corresponding documents/metadata.
- `similarity_search()` returns relevant Documents.
- `similarity_search_with_score()` returns Documents along with the
  search score.
- The meaning of the score depends on the distance/similarity metric,
  so the score should not automatically be interpreted as "higher is
  better."
- FAISS indexes can be saved locally and loaded later, allowing the
  ingestion process to be separated from query-time retrieval.